* [Snabbcheck](https://github.com/salgo60/Svenskabadplatser/blob/main/Jupyter/Snabbcheck.ipynb)

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-07-15 17:05:06.409994


In [2]:
import requests
from collections import Counter

url = "https://badplatsen.havochvatten.se/badplatsen/api/detail"

r = requests.get(url)
r.raise_for_status()

data = r.json()

print(type(data)) 
print("Antal poster:", len(data))

counter = Counter(item.get("euType") for item in data)

print(counter)

<class 'list'>
Antal poster: 2632
Counter({False: 2145, True: 487})


In [3]:
for value, count in sorted(counter.items()):
    print(f"{value}: {count}")

False: 2145
True: 487


In [4]:
import requests
from datetime import date

# -------------------------------------------------
# Hämta HaV badplatser
# -------------------------------------------------

HAV_URL = "https://badplatsen.havochvatten.se/badplatsen/api/detail"

print("Loading HaV data...")
hav = requests.get(HAV_URL).json()

print(f"HaV badplatser: {len(hav)}")

# indexera på nutsCode
hav_index = {}
for b in hav:
    nuts = b.get("nutsCode")
    if nuts:
        hav_index[nuts] = b

# -------------------------------------------------
# Hämta Wikidata
# -------------------------------------------------

query = """
SELECT ?item ?itemLabel ?EUbath WHERE {
  ?item wdt:P6104 wd:Q106774536 ;
        wdt:P17 wd:Q34 ;
        wdt:P9616 ?EUbath .

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
  }
}
ORDER BY ?itemLabel
"""

print("Loading Wikidata...")

r = requests.get(
    "https://query.wikidata.org/sparql",
    params={"query": query},
    headers={
        "Accept": "application/sparql-results+json",
        "User-Agent": "EU Bath Checker/1.0"
    },
)

wd = r.json()["results"]["bindings"]

print(f"Wikidata badplatser: {len(wd)}")

# -------------------------------------------------
# HTML
# -------------------------------------------------

today = date.today().strftime("%Y%m%d")
filename = f"EUBathSv{today}.html"

ok = 0
wrong = 0
missing = 0

rows = []

for row in wd:

    qid = row["item"]["value"].split("/")[-1]
    label = row["itemLabel"]["value"]
    eu = row["EUbath"]["value"]

    wd_link = f"https://www.wikidata.org/wiki/{qid}"
    hav_link = f"https://badplatsen.havochvatten.se/badplatsen/karta/#/bath/{eu}"

    if eu not in hav_index:
        colour = "gray"
        status = "Missing in HaV"
        missing += 1

    else:
        euType = hav_index[eu]["euType"]

        if euType:
            colour = "green"
            status = "OK"
            ok += 1
        else:
            colour = "red"
            status = "euType = false"
            wrong += 1

    rows.append(f"""
<li style="color:{colour}">
<a href="{wd_link}">{label}</a>
&nbsp;(<a href="{hav_link}">{eu}</a>)
&nbsp;- {status}
</li>
""")

# -------------------------------------------------
# skriv HTML
# -------------------------------------------------

html = f"""
<!DOCTYPE html>
<html lang="sv">
<head>
<meta charset="utf-8">
<title>EU Bath Sweden</title>
<style>
body {{
    font-family: Arial;
}}
li {{
    margin:4px;
}}
</style>
</head>

<body>

<h1>EU Bath Sweden</h1>

<p>
Datum: {today}<br>
Wikidata: {len(wd)}<br>
HaV: {len(hav)}<br><br>

<b style="color:green">OK:</b> {ok}<br>
<b style="color:red">euType=false:</b> {wrong}<br>
<b style="color:gray">Saknas i HaV:</b> {missing}
</p>

<hr>

<ul>

{''.join(rows)}

</ul>

</body>
</html>
"""

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print()
print("Finished.")
print(filename)

Loading HaV data...
HaV badplatser: 2632
Loading Wikidata...
Wikidata badplatser: 806

Finished.
EUBathSv20260715.html


## Quickstatement städning

In [5]:
qs = []

for row in wd:

    qid = row["item"]["value"].split("/")[-1]
    eu = row["EUbath"]["value"]

    if eu not in hav_index:
        continue

    if hav_index[eu]["euType"] is False:

        qs.append(
            f'{qid}\tP9616\t"{eu}"\tR-\tP2241\tQ107348938'
        )

with open(f"DeprecateP9616_{today}.qs", "w", encoding="utf-8") as f:
    f.write("\n".join(qs))

print(f"Skrev {len(qs)} QuickStatements")

Skrev 335 QuickStatements


In [6]:
print("\nMissing in HaV:\n")

for row in wd:

    eu = row["EUbath"]["value"]

    if eu not in hav_index:
        print(
            row["item"]["value"].split("/")[-1],
            eu,
            row["itemLabel"]["value"]
        )


Missing in HaV:

Q106698142 SE0241880000004167 Alnängsbadet
Q106710006 SE0812403000003125 Angsjöns badplats
Q106711983 SE0110123000004305 Basets badplats
Q106708090 SE0A21471000001028 Björkkullasand
Q106653459 SE0110120000004476 Björkviks havsbad
Q106711878 SE0622061000004018 Gladtjärn, badplats
Q106709063 SE0622082000002419 Hyen, badplats
Q106707705 SE0A21484000000571 Kolleröds badplats
Q106706969 SE0A21402000000096 Kåsjöbadet
Q106709259 SE0241880000002642 Lången
Q106708466 SE0910683000001501 Näsudden västra badplats
Q10660811 SE0712284000003065 Salusands havsbad
Q106709400 SE0A21494000002856 Villa Giacomina


## Städa dom som inte finns i Hav 
Sätt depreceated Q54554025 Källa saknas

In [7]:
missing_qs = []

for row in wd:

    qid = row["item"]["value"].split("/")[-1]
    eu = row["EUbath"]["value"]

    if eu not in hav_index:

        missing_qs.append(
            f'{qid}\tP9616\t"{eu}"\tR-\tP2241\tQ54554025'
        )

with open(f"MissingInHaV_DeprecateP9616_{today}.qs", "w", encoding="utf-8") as f:
    f.write("\n".join(missing_qs))

print(f"Skrev {len(missing_qs)} QuickStatements")

Skrev 13 QuickStatements


## Skapa ny rapport efter städning 

In [25]:
SPARQLafter = ''' 
SELECT
  ?item
  ?itemLabel
  ?statement
  ?EUbath
  ?rank
  ?reason
  ?reasonLabel 
  ?HaV
WHERE {

  ?item p:P9616 ?statement .
  ?statement ps:P9616 ?EUbath .
  ?statement wikibase:rank ?rank .

  OPTIONAL {
    ?statement pq:P2241 ?reason .
  }

  ?item wdt:P6104 wd:Q106774536 ;
        wdt:P17 wd:Q34 .

  BIND(IRI(CONCAT(
  "https://badplatsen.havochvatten.se/badplatsen/karta/#/bath/",
  ?EUbath
     )) AS ?HaV)

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
    ?item rdfs:label ?itemLabel .
    ?reason rdfs:label ?reasonLabel .
  }
}
ORDER BY ?itemLabel
'''

In [26]:
print("Loading Wikidata...")

r = requests.get(
    "https://query.wikidata.org/sparql",
    params={"query": SPARQLafter},
    headers={
        "Accept": "application/sparql-results+json",
        "User-Agent": "EU Bath Checker/1.0"
    },
)

wd = r.json()["results"]["bindings"]

print(f"Wikidata badplatser: {len(wd)}")

Loading Wikidata...
Wikidata badplatser: 2686


In [32]:
from collections import Counter

ok_rows = []
deprecated_rows = []
missing_rows = []

summary = Counter()

for row in wd:
    if qid == "Q106708004":
        print(row["rank"]["value"])
        print(rank)
        print(repr(rank))

    qid = row["item"]["value"].split("/")[-1]
    label = row["itemLabel"]["value"]
    eu = row["EUbath"]["value"]
    rank = row["rank"]["value"].split("#")[-1].replace("Rank", "")

    reason = ""
    if "reasonLabel" in row:
        reason = row["reasonLabel"]["value"]

    wd_link = f"https://www.wikidata.org/wiki/{qid}"
    hav_link = f"https://badplatsen.havochvatten.se/badplatsen/karta/#/bath/{eu}"

    # Saknas i HaV
    if eu not in hav_index:

        summary["missing"] += 1

        missing_rows.append(f"""
<tr class="missing">
<td><a href="{wd_link}">{qid}</a></td>
<td>{label}</td>
<td>{eu}</td>
<td>{rank}</td>
<td>{reason}</td>
<td>Missing in HaV</td>
</tr>
""")
        continue

    rank_icon = {
    "Preferred": "⭐",
    "Normal": "🟢",
    "Deprecated": "🟡"
        }.get(rank, "")
    
    # Deprecated
    if rank == "Deprecated":

        summary["deprecated"] += 1

        deprecated_rows.append(f"""
<tr class="deprecated">
<td><a href="{wd_link}">{qid}</a></td>
<td>{label}</td>
<td>{eu}</td>
<td>{rank_icon} {rank}</td>
<td>{reason}</td>
<td><a href="{hav_link}">HaV</a></td>
</tr>
""")

    # Normal eller Preferred
    else:

        summary["ok"] += 1

        ok_rows.append(f"""
<tr class="ok">
<td><a href="{wd_link}">{qid}</a></td>
<td>{label}</td>
<td>{eu}</td>
<td>{rank}</td>
<td></td>
<td><a href="{hav_link}">HaV</a></td>
</tr>
""")

http://wikiba.se/ontology#DeprecatedRank
Deprecated
'Deprecated'


In [34]:
print(summary)

print("OK:", len(ok_rows))
print("Deprecated:", len(deprecated_rows))
print("Missing:", len(missing_rows))

Counter({'deprecated': 2045, 'ok': 459, 'missing': 182})
OK: 459
Deprecated: 2045
Missing: 182


In [35]:
html = f"""<!DOCTYPE html>
<html lang="sv">
<head>
<meta charset="utf-8">
<title>EU Bath Sweden after cleanup</title>

<style>
body {{
    font-family: Arial, Helvetica, sans-serif;
    margin: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-bottom: 30px;
}}

th, td {{
    border: 1px solid #ccc;
    padding: 6px;
    text-align: left;
}}

th {{
    background: #f0f0f0;
}}

.ok {{
    background: #e8ffe8;
}}

.deprecated {{
    background: #fff3cd;
}}

.missing {{
    background: #f8d7da;
}}

</style>
</head>

<body>

<h1>EU Bath Sweden – after cleanup</h1>

<p>
Generated: {today}
</p>

<h2>Summary</h2>

<table>
<tr><th>Status</th><th>Count</th></tr>
<tr><td>🟢 Normal rank</td><td>{summary["ok"]}</td></tr>
<tr><td>🟡 Deprecated rank</td><td>{summary["deprecated"]}</td></tr>
<tr><td>🔴 Missing in HaV</td><td>{summary["missing"]}</td></tr>
</table>


<h2>🟢 Active EU bathing waters</h2>

<table>

<tr>
<th>QID</th>
<th>Name</th>
<th>P9616</th>
<th>Rank</th>
<th>Reason</th>
<th>HaV</th>
</tr>

{''.join(ok_rows)}

</table>


<h2>🟡 Deprecated</h2>

<table>

<tr>
<th>QID</th>
<th>Name</th>
<th>P9616</th>
<th>Rank</th>
<th>Reason</th>
<th>HaV</th>
</tr>

{''.join(deprecated_rows)}

</table>


<h2>🔴 Missing in HaV</h2>

<table>

<tr>
<th>QID</th>
<th>Name</th>
<th>P9616</th>
<th>Rank</th>
<th>Reason</th>
<th>Status</th>
</tr>

{''.join(missing_rows)}

</table>

</body>
</html>
"""

In [37]:
print(f"""
HaV bathing waters : {len(hav)}
Wikidata items     : {len(wd)}

Normal rank        : {summary['ok']}
Deprecated         : {summary['deprecated']}
Missing in HaV     : {summary['missing']}
""")


HaV bathing waters : 2632
Wikidata items     : 2686

Normal rank        : 459
Deprecated         : 2045
Missing in HaV     : 182



In [38]:
filename = f"EUBathSv{today}_afterclean.html"

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Created {filename}")

Created EUBathSv20260715_afterclean.html


In [39]:
for row in wd:
    if row["item"]["value"].endswith("Q106708004"):
        print(row)

{'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q106708004'}, 'statement': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/statement/Q106708004-7476D7AC-FC3A-48D1-A70A-5B45F84AAEBA'}, 'EUbath': {'type': 'literal', 'value': 'SE0A21494000000915'}, 'rank': {'type': 'uri', 'value': 'http://wikiba.se/ontology#DeprecatedRank'}, 'reason': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q107348938'}, 'itemLabel': {'xml:lang': 'sv', 'type': 'literal', 'value': 'Arkens badplats'}, 'reasonLabel': {'xml:lang': 'sv', 'type': 'literal', 'value': 'ej EU-bad'}, 'HaV': {'type': 'uri', 'value': 'https://badplatsen.havochvatten.se/badplatsen/karta/#/bath/SE0A21494000000915'}}


In [40]:
print(rank)

Deprecated
